In [1]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

In [2]:
#importing the required libraries
from transformers import AutoTokenizer, AutoModelForCausalLM  
from peft import LoraConfig,get_peft_model
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig



In [5]:
#loading the model
model_name="meta-llama/Llama-3.2-3B"
device="auto"
model=AutoModelForCausalLM.from_pretrained(model_name)
tokenizer=AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [6]:
#loading the dataset
dataset=load_dataset("MaryWambo/formatted_dataset", split="train")

def formatting_func(example):
    text = (f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"{example['input']}<|eot_id|><|start_header_id|>"
            f"assistant<|end_header_id|>\n\n{example['output']}<|eot_id|>")
    return {"text" : text}

dataset = dataset.map(formatting_func)
print(dataset[7])





formatted_dataset.json:   0%|          | 0.00/8.32M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30431 [00:00<?, ? examples/s]

{'input': 'Irrigate if necessary.', 'output': 'Ĩtĩrĩria maĩ angĩkorũo hena bata.', 'text': '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nIrrigate if necessary.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nĨtĩrĩria maĩ angĩkorũo hena bata.<|eot_id|>'}


In [8]:
#defining the PEFT configuration(LoRA)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj","k_proj","gate_proj","up_proj","down_proj"] 
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



trainable params: 10,780,672 || all params: 3,223,530,496 || trainable%: 0.3344


/users/mkariuki/miniconda3/envs/lenv/lib/python3.12/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/users/mkariuki/miniconda3/envs/lenv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [7]:
#defining traning parameters
training_args=SFTConfig(
    output_dir="finetunned_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    dataset_text_field = "text",
    label_names=["labels"],
    num_train_epochs=2,
    logging_steps=1,
    max_seq_length=512,
    weight_decay=0.01,
    optim = "adamw_torch",
    warmup_steps=5,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    report_to="none", 
)




In [8]:
#intializing the trainer
trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,

)                   

In [12]:
# !nvidia-smi


In [14]:
# print(torch.cuda.device_count())  


In [9]:
trainer.train()

Step,Training Loss
1,5.518700
2,6.055200
3,5.510000
4,6.251300
5,5.241700
6,5.361500
7,3.360300
8,4.090000
9,3.842300
10,4.002600


TrainOutput(global_step=7608, training_loss=1.9327859023956093, metrics={'train_runtime': 5901.128, 'train_samples_per_second': 10.314, 'train_steps_per_second': 1.289, 'total_flos': 1.1528818156436275e+17, 'train_loss': 1.9327859023956093})

In [1]:
# visualizing the performance of the model
import matplotlib.pyplot as plt
import pandas as pd

# Extracting loss and steps from trainer logs
log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

#sampling the data
subsample_interval = 100
subsampled_steps = steps[::subsample_interval]
subsampled_loss = losses[::subsample_interval]
plt.plot(subsampled_steps, subsampled_loss, color='blue', marker='o', linestyle='dashed')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.grid()


In [ ]:
#saving the model to the hub
model_name = "MaryWambo/final_trained_weights"

# Save model and tokenizer locally
model.save_pretrained(model_name)
tokenizer.save_pretrained(model_name)

model.push_to_hub(model_name, token=token)
tokenizer.push_to_hub(model_name, token=token)

In [ ]:
# inferencing
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Loading the  fine-tuned model
model_path = "finetunned_model"
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_path)

def create_prompt(input_text):
    return (f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"{input_text}<|eot_id|><|start_header_id|>"
            f"assistant<|end_header_id|>\n\n")

input_text = "harvesting is done on the first month"
prompt = create_prompt(input_text)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generating response
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
response = full_text[len(prompt):].split('<|eot_id|>')[0].strip()

print("Input:", input_text)
print("Response:", response)

In [ ]:
#evaluating using BLEU score
